In [12]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [2]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models


In [3]:
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
tail = [A.Resize(224, 224), A.Normalize(mean, std), ToTensorV2()]

configs={
    "baseline": A.Compose([*tail]),
    "flip": A.Compose([A.HorizontalFlip(p=0.5), *tail]),
    "rotate": A.Compose([A.Affine(rotate=(-15, 15), p=0.5), *tail]),
    "colorjit": A.Compose([A.ColorJitter(
    brightness_range=(0.8, 1.2),
    contrast_range=(0.8, 1.2),
    saturation_range=(0.8, 1.2),
    hue_range=(-0.1, 0.1),
    p=0.5), *tail]),
    "randomcrop": A.Compose([A.PadIfNeeded(40, 40, border_mode=0), 
                             A.RandomCrop(32, 32), *tail]),
    "cutout": A.Compose([A.CoarseDropout(num_holes_range=(1, 1),
                                         hole_height_range=(0.25, 0.25),
                                         hole_width_range=(0.25, 0.25),
                                         fill=0, p=0.5), *tail]),
}
val_tf=A.Compose([*tail])

In [4]:
import numpy as np

class AlbAdapter:
    def __init__(self, tf): self.tf = tf
    def __call__(self, img):                      # img: PIL
        return self.tf(image=np.array(img))["image"]

In [7]:
import random, numpy as np
from torch.utils.data import Subset

def run_one(train_tf, val_tf, epochs=3, seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)   

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(device)

    model = models.efficientnet_b0(weights="DEFAULT")
    model.classifier[1] = nn.Linear(1280, 10)
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
  
    root = '/Users/hwanghyunjin/Desktop/cv-portfolio/week1/data'
    
    train_data = datasets.CIFAR10(root, train=True,  download=True, transform=AlbAdapter(train_tf))
    train_data = Subset(train_data, range(10000))
    
    test_data  = datasets.CIFAR10(root, train=False, transform=AlbAdapter(val_tf))
    
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_data,  batch_size=64)
    
    def train(model, loader, criterion, optimizer, epoch):
        model.train()
        total_loss = 0
       
        for batch_idx, (data, target) in enumerate(loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f'Epoch {epoch}: Loss = {total_loss/len(loader):.4f}')
        return total_loss / len(loader)

    def evaluate(model, loader):
        model.eval()
        correct = 0
        
        with torch.no_grad():
            for data, target in loader:
                data, target = data.to(device), target.to(device)  
                output = model(data)
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()
                
        acc = 100. * correct / len(loader.dataset)
        print(f'Accuracy: {acc:.2f}%')
        return acc

    for epoch in range(1, epochs+1):          
        train(model, train_loader, criterion, optimizer, epoch)

    train_acc = evaluate(model, train_loader)  
    val_acc   = evaluate(model, test_loader)
       
    return train_acc, val_acc               

In [15]:
import pandas as pd

rows = []
for name, tf in configs.items():
    print(f"\n===== {name} =====")
    tr, va = run_one(tf, val_tf, epochs=2)
    rows.append({"aug": name, "train_acc": tr, "val_acc": va, "gap": round(tr - va, 2)})

df = pd.DataFrame(rows).sort_values("val_acc", ascending=False)
print("\n", df.to_markdown(index=False))


===== baseline =====
mps
Epoch 1: Loss = 0.5967
Epoch 2: Loss = 0.2694
Accuracy: 94.77%
Accuracy: 86.88%

===== flip =====
mps
Epoch 1: Loss = 0.5929
Epoch 2: Loss = 0.2901
Accuracy: 95.45%
Accuracy: 88.40%

===== rotate =====
mps
Epoch 1: Loss = 0.6473
Epoch 2: Loss = 0.3341
Accuracy: 93.19%
Accuracy: 86.95%

===== colorjit =====
mps
Epoch 1: Loss = 0.6230
Epoch 2: Loss = 0.2906
Accuracy: 95.02%
Accuracy: 87.81%

===== randomcrop =====
mps
Epoch 1: Loss = 0.6288
Epoch 2: Loss = 0.3315
Accuracy: 93.12%
Accuracy: 86.59%

===== cutout =====
mps
Epoch 1: Loss = 0.6383
Epoch 2: Loss = 0.3096
Accuracy: 93.57%
Accuracy: 87.14%


ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

In [16]:
pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [18]:
print(df)

          aug  train_acc  val_acc   gap
1        flip      95.45    88.40  7.05
3    colorjit      95.02    87.81  7.21
5      cutout      93.57    87.14  6.43
2      rotate      93.19    86.95  6.24
0    baseline      94.77    86.88  7.89
4  randomcrop      93.12    86.59  6.53
